In [ ]:
from dotenv import load_dotenv

load_dotenv()

In [ ]:
from llm_judge import create_log_judge_agent, format_judge_prompt

In [ ]:
import json

with open("evals_run_2026_03_06_results.json", "r") as f:
    results = json.load(f)

print(f"Loaded {len(results)} results.")

In [ ]:
judge = create_log_judge_agent()

In [ ]:
row = results[0]

In [ ]:
prompt = format_judge_prompt(row)

In [ ]:
eval_result = await judge.run(prompt)
eval_output = eval_result.output

In [ ]:
print(eval_output.label)
print(eval_output.reasoning)

In [ ]:
async def run_eval(row):
    prompt = format_judge_prompt(row)
    eval_result = await judge.run(prompt)
    eval_output = eval_result.output
    return eval_output

In [ ]:
from tqdm.auto import tqdm

In [ ]:
eval_results = []

for row in tqdm(results):
    eval_output = await run_eval(row)
    eval_results.append({
        'row': row,
        'label': eval_output.label,
        'reasoning': eval_output.reasoning
    })

In [ ]:
rows = []

for result in eval_results:
    row = {
        'question': result['row']['input']['question'],
        'llm_answer': result['row']['rag_response']['answer'],
        'human_label': result['row']['label'],
        'human_comment': result['row']['comments'],
        'llm_label': result['label'],
        'llm_reasoning': result['reasoning']
    }
    rows.append(row)

In [ ]:
import pandas as pd
df_rows = pd.DataFrame(rows)

In [ ]:
df_rows.llm_label.value_counts()

In [ ]:
df_rows.human_label.value_counts()

In [ ]:
(df_rows.llm_label == df_rows.human_label).mean()

In [ ]:
df_disagreement = df_rows[df_rows.llm_label != df_rows.human_label]

In [ ]:
df_disagreement